# 🏗️ Infrastructure & Deployment — Hands-On Lab## Build production-ready LLM infrastructure patterns you can run NOW.**What this notebook does:**- Multi-provider gateway with automatic fallback- Model routing (cheap model for simple, expensive for complex)- Streaming (word-by-word output)- Retry with exponential backoff + circuit breaker- Quantization size calculator- Platform cost comparison at YOUR scale- Production health check wrapper**Setup:**```pip install openai litellm```**Cost:** ~$0.20 to run everything**Note:** Programs 1-7 run on ANY laptop. No GPU, no Docker, no Kubernetes. These are the PATTERNS you'll use when you deploy — tested locally first.

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================

# pip install openai litellm
from openai import OpenAI
import time, json, random, math
from collections import defaultdict

client = OpenAI()

def ask(prompt, model="gpt-4o-mini", temperature=0):
    start = time.time()
    r = client.chat.completions.create(
        model=model, temperature=temperature,
        messages=[{"role":"user","content":prompt}])
    latency = (time.time() - start) * 1000
    return r.choices[0].message.content.strip(), r.usage.total_tokens, latency

answer, tokens, ms = ask("Say 'Infra ready' in 2 words.")
print(f"Setup complete! {answer} ({ms:.0f}ms)")

---# 1️⃣ Multi-Provider Gateway (Never Depend on One Provider)**Analogy:** You order restaurant supplies from one distributor. They have a truck breakdown. Your restaurant shuts down. Stupid! You need backup distributors. A gateway routes across multiple LLM providers with automatic failover.**What we build:** A gateway that tries Provider A → if it fails → tries Provider B → if that fails → tries Provider C. User never notices.

In [ ]:
# ============================================================
# GATEWAY: One interface for all providers. Auto-failover.
# Like a hotel concierge who handles all external calls for you.
# ============================================================

class LLMGateway:
    """
    Single entry point for all LLM calls.
    Handles: fallback, retry, logging, cost tracking.
    """
    
    def __init__(self):
        self.call_log = []
        self.provider_health = defaultdict(lambda: {"success": 0, "fail": 0})
    
    def call(self, prompt, providers=None):
        """
        Try providers in order. First success wins.
        If all fail, return graceful error.
        """
        
        if providers is None:
            providers = [
                {"name": "primary",   "model": "gpt-4o-mini"},
                {"name": "backup",    "model": "gpt-4o-mini"},  # In prod: different provider
                {"name": "emergency", "model": "gpt-4o-mini"},  # In prod: local model
            ]
        
        for provider in providers:
            try:
                start = time.time()
                
                # Simulate occasional failures (20% chance for demo)
                if random.random() < 0.2 and provider["name"] != "emergency":
                    raise Exception(f"Simulated {provider['name']} outage")
                
                answer, tokens, latency = ask(prompt, model=provider["model"])
                
                # Log success
                self.provider_health[provider["name"]]["success"] += 1
                self.call_log.append({
                    "provider": provider["name"],
                    "tokens": tokens,
                    "latency_ms": latency,
                    "status": "success"
                })
                
                return answer, provider["name"]
                
            except Exception as e:
                # Log failure and try next provider
                self.provider_health[provider["name"]]["fail"] += 1
                self.call_log.append({
                    "provider": provider["name"],
                    "status": "failed",
                    "error": str(e)[:50]
                })
                print(f"    ⚡ {provider['name']} failed: {str(e)[:40]}... trying next...")
        
        return "Service temporarily unavailable. Please try again.", "none"
    
    def health_report(self):
        print("\n  PROVIDER HEALTH:")
        for name, stats in self.provider_health.items():
            total = stats["success"] + stats["fail"]
            if total > 0:
                uptime = stats["success"] / total * 100
                print(f"    {name:12s}: {uptime:5.1f}% uptime ({stats['success']}/{total} calls)")
        
        if self.call_log:
            successes = [c for c in self.call_log if c["status"] == "success"]
            if successes:
                by_provider = defaultdict(int)
                for c in successes:
                    by_provider[c["provider"]] += 1
                print(f"\n  TRAFFIC DISTRIBUTION:")
                for provider, count in by_provider.items():
                    pct = count / len(successes) * 100
                    bar = "█" * int(pct / 5)
                    print(f"    {provider:12s}: {bar} {pct:.0f}%")


# ---------- TEST ----------

print("MULTI-PROVIDER GATEWAY")
print("=" * 60)
print("  Simulating 10 calls with 20% random failures...\n")

gateway = LLMGateway()

for i in range(10):
    query = random.choice([
        "What is 2+2?", "Capital of France?", "Refund policy?",
        "Pro plan price?", "Hello!", "Thanks!",
    ])
    answer, provider = gateway.call(query)
    print(f"  Call {i+1:2d}: [{provider:9s}] {query:25s} → {answer[:30]}...")

gateway.health_report()

print("\n💡 User never sees the failover. They just get an answer.")
print("💡 In production: use LiteLLM as the gateway proxy.")
print("💡 Primary=GPT-4o, Backup=Claude, Emergency=local Llama.")

---# 2️⃣ Model Routing (Cheap for Simple, Expensive for Complex)**Analogy:** A hospital ER. Paper cut → nurse ($5). Broken arm → doctor ($50). Heart attack → surgeon ($500). You don't pay surgeon rates for a paper cut.**What we build:** Classify query complexity → route to the right model → track cost savings.

In [ ]:
# ============================================================
# MODEL ROUTING: Simple queries → cheap model. Complex → expensive.
# 70% of queries are simple. Routing saves ~60% of costs.
# ============================================================

class ModelRouter:
    """Route queries to the right model based on complexity."""
    
    # Model pricing per 1M tokens
    MODELS = {
        "simple":  {"model": "gpt-4o-mini",  "price": 0.15,  "label": "Nurse"},
        "complex": {"model": "gpt-4o",       "price": 5.0,   "label": "Surgeon"},
    }
    
    COMPLEX_SIGNALS = [
        "compare", "analyze", "design", "explain how", "tradeoffs",
        "step by step", "architecture", "evaluate", "recommend",
        "pros and cons", "difference between", "why should",
    ]
    
    def __init__(self):
        self.routing_log = []
    
    def classify(self, query):
        """Determine if a query is simple or complex."""
        query_lower = query.lower()
        
        # Rule 1: complex keywords
        if any(signal in query_lower for signal in self.COMPLEX_SIGNALS):
            return "complex"
        
        # Rule 2: long queries are usually complex
        if len(query.split()) > 30:
            return "complex"
        
        # Rule 3: questions with multiple parts
        if query.count("?") > 1 or " and " in query_lower:
            return "complex"
        
        return "simple"
    
    def route(self, query):
        """Classify and route to the right model."""
        complexity = self.classify(query)
        model_info = self.MODELS[complexity]
        
        answer, tokens, latency = ask(query, model=model_info["model"])
        cost = tokens / 1_000_000 * model_info["price"]
        
        self.routing_log.append({
            "query": query[:40],
            "complexity": complexity,
            "model": model_info["model"],
            "tokens": tokens,
            "cost": cost,
            "latency": latency,
        })
        
        return answer, complexity, cost
    
    def savings_report(self):
        """Compare routed cost vs all-expensive cost."""
        actual_cost = sum(r["cost"] for r in self.routing_log)
        all_expensive = sum(
            r["tokens"] / 1_000_000 * self.MODELS["complex"]["price"]
            for r in self.routing_log
        )
        savings = all_expensive - actual_cost
        savings_pct = (savings / all_expensive * 100) if all_expensive > 0 else 0
        
        simple_count = sum(1 for r in self.routing_log if r["complexity"] == "simple")
        complex_count = sum(1 for r in self.routing_log if r["complexity"] == "complex")
        
        print(f"\n  ROUTING REPORT ({len(self.routing_log)} queries):")
        print(f"  {'─' * 45}")
        print(f"    Simple (→ mini):  {simple_count} queries")
        print(f"    Complex (→ 4o):   {complex_count} queries")
        print(f"    Routed cost:      ${actual_cost:.4f}")
        print(f"    All-GPT-4o cost:  ${all_expensive:.4f}")
        print(f"    SAVINGS:          ${savings:.4f} ({savings_pct:.0f}%)")


# ---------- TEST ----------

print("MODEL ROUTING")
print("=" * 60)

router = ModelRouter()

queries = [
    "What is 2+2?",
    "What is the refund policy?",
    "Compare PostgreSQL vs MongoDB for a social media app with 10M users",
    "Hello!",
    "How much is the Pro plan?",
    "Design a microservices architecture for an e-commerce platform",
    "Thanks for your help!",
    "Explain how transformers work step by step",
    "What time do you close?",
    "Analyze the tradeoffs between REST and GraphQL APIs",
]

print()
for q in queries:
    answer, complexity, cost = router.route(q)
    emoji = "🟢" if complexity == "simple" else "🔴"
    print(f"  {emoji} [{complexity:7s}] ${cost:.5f} | {q[:50]}...")

router.savings_report()

print("\n💡 Simple queries (70%) go to cheap model. Zero quality loss.")
print("💡 Complex queries (30%) still get the best model.")
print("💡 At 100K queries/day: saves $1000+/day.")

---# 3️⃣ Streaming (Word-by-Word Output)**Analogy:** Without streaming: you stare at a blank screen for 5 seconds, then ALL the text appears at once. WITH streaming: the first word appears in 300ms and the rest flows in. Users perceive it as 3-5x faster (same total time).**What we build:** Streaming response + measure first-token latency vs total latency.

In [ ]:
# ============================================================
# STREAMING: Words appear one at a time
# First token latency is what users FEEL
# ============================================================

def streaming_demo(prompt, model="gpt-4o-mini"):
    """
    Stream a response word by word.
    Measure: time to first token vs total time.
    """
    
    start = time.time()
    first_token_time = None
    full_response = ""
    token_count = 0
    
    # Create streaming response
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,  # This is the magic flag
    )
    
    print("  Streaming: ", end="", flush=True)
    
    for chunk in stream:
        if chunk.choices[0].delta.content:
            token = chunk.choices[0].delta.content
            full_response += token
            token_count += 1
            
            # Record first token time
            if first_token_time is None:
                first_token_time = (time.time() - start) * 1000
            
            # Print each token as it arrives
            print(token, end="", flush=True)
            time.sleep(0.01)  # Tiny delay so you can see the streaming effect
    
    total_time = (time.time() - start) * 1000
    
    print()
    return first_token_time, total_time, token_count


def non_streaming_demo(prompt, model="gpt-4o-mini"):
    """Normal (non-streaming) call. User waits for EVERYTHING."""
    start = time.time()
    answer, tokens, _ = ask(prompt, model=model)
    total_time = (time.time() - start) * 1000
    return total_time, tokens


# ---------- COMPARE ----------

prompt = "Explain what cloud computing is in 3 sentences."

print("STREAMING vs NON-STREAMING")
print("=" * 60)

# Non-streaming first
print("\n  NON-STREAMING (user sees nothing, then everything):")
print("  Waiting", end="", flush=True)
ns_time, ns_tokens = non_streaming_demo(prompt)
print(f"  Total wait: {ns_time:.0f}ms (user stared at blank screen the whole time)")

# Streaming
print(f"\n  STREAMING (words appear one by one):")
first_ms, total_ms, token_count = streaming_demo(prompt)

print(f"\n  COMPARISON:")
print(f"  {'─' * 45}")
print(f"    First word appeared:  {first_ms:>6.0f} ms (streaming)")
print(f"    vs blank screen for:  {ns_time:>6.0f} ms (non-streaming)")
print(f"    Total time:           {total_ms:>6.0f} ms (same either way)")
print(f"    Perceived speedup:    {ns_time/max(first_ms,1):.1f}x faster feeling")

print("\n💡 Total time is the SAME. But first word in 300ms vs 2000ms = huge UX difference.")
print("💡 Target: first token < 500ms. Users start reading immediately.")
print("💡 stream=True is the ONLY change needed. One flag.")

---# 4️⃣ Retry with Backoff + Circuit Breaker**Retry:** Request fails? Wait 1s, try again. Fails? Wait 2s. Again? Wait 4s. Each wait doubles (exponential backoff). Add randomness (jitter) so 1000 users don't all retry at the exact same second.**Circuit Breaker:** After 5 failures, STOP trying (circuit opens). Wait 60 seconds. Try ONE request (half-open). If it works, resume. Like a fuse that protects your house.

In [ ]:
# ============================================================
# RETRY + CIRCUIT BREAKER: Handle provider failures gracefully
# ============================================================

class RetryWithBackoff:
    """
    Exponential backoff: 1s → 2s → 4s with random jitter.
    Prevents thundering herd (1000 users retrying at once).
    """
    
    def __init__(self, max_retries=3, base_delay=1.0):
        self.max_retries = max_retries
        self.base_delay = base_delay
    
    def execute(self, func, *args, **kwargs):
        for attempt in range(self.max_retries + 1):
            try:
                result = func(*args, **kwargs)
                if attempt > 0:
                    print(f"    ✅ Succeeded on attempt {attempt + 1}")
                return result
            except Exception as e:
                if attempt == self.max_retries:
                    print(f"    ❌ All {self.max_retries + 1} attempts failed")
                    raise
                
                # Exponential backoff with jitter
                delay = self.base_delay * (2 ** attempt) + random.uniform(0, 0.5)
                print(f"    ⚡ Attempt {attempt + 1} failed: {str(e)[:30]}...")
                print(f"    ⏳ Waiting {delay:.1f}s before retry...")
                time.sleep(min(delay, 2.0))  # Cap at 2s for demo


class CircuitBreaker:
    """
    CLOSED: normal operation, requests flow through.
    OPEN: provider is down, immediately return fallback.
    HALF-OPEN: testing if provider recovered.
    """
    
    def __init__(self, failure_threshold=3, recovery_timeout=10):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failures = 0
        self.state = "CLOSED"
        self.last_failure_time = 0
    
    def call(self, func, fallback_func, *args, **kwargs):
        # If circuit is OPEN, check if we should try recovery
        if self.state == "OPEN":
            time_since_failure = time.time() - self.last_failure_time
            if time_since_failure > self.recovery_timeout:
                self.state = "HALF-OPEN"
                print(f"    🔄 Circuit HALF-OPEN: testing recovery...")
            else:
                print(f"    ⚡ Circuit OPEN: immediate fallback ({self.recovery_timeout - time_since_failure:.0f}s until retry)")
                return fallback_func(*args, **kwargs)
        
        try:
            result = func(*args, **kwargs)
            
            if self.state == "HALF-OPEN":
                self.state = "CLOSED"
                self.failures = 0
                print(f"    ✅ Circuit CLOSED: provider recovered!")
            
            return result
            
        except Exception as e:
            self.failures += 1
            self.last_failure_time = time.time()
            
            if self.failures >= self.failure_threshold:
                self.state = "OPEN"
                print(f"    🔴 Circuit OPEN after {self.failures} failures!")
            else:
                print(f"    ⚡ Failure {self.failures}/{self.failure_threshold}")
            
            return fallback_func(*args, **kwargs)


# ---------- TEST RETRY ----------

print("RETRY WITH EXPONENTIAL BACKOFF")
print("=" * 60)

retrier = RetryWithBackoff(max_retries=3, base_delay=0.5)

attempt_count = 0
def flaky_api(prompt):
    global attempt_count
    attempt_count += 1
    if attempt_count <= 2:  # Fail first 2 times
        raise Exception("Connection timeout")
    answer, _, _ = ask(prompt)
    return answer

print("\n  Calling a flaky API (fails 2 times, succeeds on 3rd):")
attempt_count = 0
result = retrier.execute(flaky_api, "What is 2+2?")
print(f"  Result: {result}")


# ---------- TEST CIRCUIT BREAKER ----------

print(f"\n\nCIRCUIT BREAKER")
print("=" * 60)

breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=5)

def primary_provider(prompt):
    raise Exception("Provider down")

def fallback_provider(prompt):
    answer, _, _ = ask(prompt)
    return f"[FALLBACK] {answer}"

print("\n  Simulating primary provider going down:")
for i in range(5):
    result = breaker.call(primary_provider, fallback_provider, "What is 2+2?")
    print(f"  Call {i+1}: {result[:40]}...")
    print(f"  State: {breaker.state} | Failures: {breaker.failures}")
    print()

print("  Waiting for recovery timeout...")
time.sleep(1)  # Shortened for demo

print(f"\n💡 Retry: handles temporary failures (network blips).")
print(f"💡 Circuit breaker: handles prolonged outages (provider down).")
print(f"💡 Together: retry for blips, circuit breaker for outages.")
print(f"💡 ALWAYS add jitter to prevent thundering herd.")

---# 5️⃣ Quantization Calculator (How Much Memory Do You Need?)**Analogy:** A 4K movie is 50GB. Compress to 1080p: 10GB, looks 97% as good. Quantization does the same for LLMs. A 70B model goes from 140GB to 35GB with 1-3% quality loss.**What we build:** Calculate memory requirements for ANY model at ANY quantization level. Find out what fits on YOUR hardware.

In [ ]:
# ============================================================
# QUANTIZATION CALCULATOR: Know exactly what fits on your GPU
# No GPU needed to run this — it's pure math
# ============================================================

def quantization_calculator(
    model_name,
    total_params_billions,
    your_gpu_memory_gb=None,
):
    """
    Calculate memory needs at each quantization level.
    Shows what fits on common GPUs.
    """
    
    # Bytes per parameter at each precision
    precisions = {
        "FP32 (full)":     4.0,
        "FP16 (half)":     2.0,
        "INT8 (8-bit)":    1.0,
        "INT4 (4-bit)":    0.5,
        "INT3 (3-bit)":    0.375,
    }
    
    # Quality retention (approximate)
    quality = {
        "FP32 (full)":     "100%",
        "FP16 (half)":     "~100%",
        "INT8 (8-bit)":    "~99%",
        "INT4 (4-bit)":    "~97%",
        "INT3 (3-bit)":    "~93%",
    }
    
    # Common GPUs
    gpus = {
        "RTX 3090":      24,
        "RTX 4090":      24,
        "A100 (40GB)":   40,
        "A100 (80GB)":   80,
        "H100 (80GB)":   80,
        "Mac M1 Pro":    16,    # Unified memory
        "Mac M2 Max":    32,
        "Mac M3 Max":    64,
        "Mac M4 Max":    128,
    }
    
    params = total_params_billions * 1_000_000_000
    
    print(f"  MODEL: {model_name} ({total_params_billions}B parameters)")
    print(f"  {'─' * 55}")
    print(f"  {'Precision':<16} {'Size':>8} {'Quality':>8} {'Fits on':>25}")
    print(f"  {'─' * 55}")
    
    for precision, bytes_per_param in precisions.items():
        size_bytes = params * bytes_per_param
        size_gb = size_bytes / (1024**3)
        
        # Add ~20% overhead for KV cache and runtime
        runtime_gb = size_gb * 1.2
        
        # Find smallest GPU it fits on
        fits_on = []
        for gpu_name, gpu_mem in sorted(gpus.items(), key=lambda x: x[1]):
            if runtime_gb <= gpu_mem:
                fits_on.append(gpu_name)
        
        fit_str = fits_on[0] if fits_on else "Multi-GPU needed"
        q = quality[precision]
        
        marker = " ← SWEET SPOT" if "4-bit" in precision else ""
        print(f"  {precision:<16} {size_gb:>6.1f}GB {q:>8} {fit_str:>25}{marker}")
    
    if your_gpu_memory_gb:
        print(f"\n  YOUR GPU: {your_gpu_memory_gb}GB")
        best_precision = None
        for precision, bytes_per_param in precisions.items():
            size_gb = (params * bytes_per_param) / (1024**3) * 1.2
            if size_gb <= your_gpu_memory_gb:
                best_precision = precision
                best_size = size_gb
                break
        
        if best_precision:
            print(f"  Best fit: {best_precision} ({best_size:.1f}GB)")
        else:
            print(f"  ⚠️ This model doesn't fit on your GPU at any precision")


# ---------- CALCULATE FOR POPULAR MODELS ----------

print("QUANTIZATION CALCULATOR")
print("=" * 60)
print()

models = [
    ("Llama-3-8B", 8),
    ("Llama-3-70B", 70),
    ("Llama-3-405B", 405),
    ("Mixtral 8x7B", 47),
]

for name, params in models:
    quantization_calculator(name, params)
    print()

# The golden rule
print("=" * 60)
print("  THE GOLDEN RULE:")
print("  A quantized BIG model beats a full-precision SMALL model.")
print("  70B at 4-bit (97% of a genius) > 13B at FP16 (100% of average)")
print("  Always quantize the BIGGEST model that fits your hardware.")

print("\n  YOUR HARDWARE? Try:")
print("  quantization_calculator('Llama-3-70B', 70, your_gpu_memory_gb=24)")

---# 6️⃣ Platform Cost Calculator (At YOUR Scale)**Analogy:** Choosing between meal delivery ($15/meal), restaurants ($30/meal), and cooking at home ($5/meal but you need a kitchen). The right choice depends on how many meals you eat.**What we build:** Enter YOUR query volume → see exact costs across all major platforms.

In [ ]:
# ============================================================
# PLATFORM COST COMPARISON: Find the cheapest option at YOUR scale
# ============================================================

def platform_cost_calculator(queries_per_day, avg_tokens=300):
    """
    Compare costs across platforms at YOUR query volume.
    Input: queries per day + average tokens per query.
    Output: monthly cost for each platform.
    """
    
    daily_tokens = queries_per_day * avg_tokens
    monthly_tokens = daily_tokens * 30
    
    # Platform pricing (per 1M tokens, input+output averaged)
    platforms = {
        "OpenAI GPT-4o":        {"price": 5.0,   "type": "API"},
        "OpenAI GPT-4o-mini":   {"price": 0.15,  "type": "API"},
        "Anthropic Claude 3.5": {"price": 4.5,   "type": "API"},
        "Groq (Llama-70B)":     {"price": 0.80,  "type": "API"},
        "Together (Llama-70B)": {"price": 0.90,  "type": "API"},
        "AWS Bedrock (Claude)": {"price": 4.5,   "type": "Cloud"},
        "Azure OpenAI (GPT-4o)":{"price": 5.0,   "type": "Cloud"},
        "Self-hosted (1xA100)": {"price": 0.05,  "type": "Self", "fixed_monthly": 2400},
        "Self-hosted (1x4090)": {"price": 0.03,  "type": "Self", "fixed_monthly": 300},
    }
    
    print(f"  YOUR SCALE: {queries_per_day:,} queries/day × {avg_tokens} tokens = {daily_tokens:,} tokens/day")
    print(f"  Monthly: {monthly_tokens:,.0f} tokens\n")
    
    print(f"  {'Platform':<28} {'Monthly Cost':>12} {'Per Query':>10} {'Type':>8}")
    print(f"  {'─' * 62}")
    
    results = []
    for name, info in platforms.items():
        monthly_cost = (monthly_tokens / 1_000_000) * info["price"]
        
        # Add fixed costs for self-hosted
        if "fixed_monthly" in info:
            monthly_cost = max(monthly_cost, info["fixed_monthly"])
        
        per_query = monthly_cost / (queries_per_day * 30)
        results.append((monthly_cost, name, per_query, info["type"]))
    
    # Sort by cost
    results.sort()
    
    cheapest = results[0][1]
    for monthly, name, per_query, ptype in results:
        marker = " ← CHEAPEST" if name == cheapest else ""
        print(f"  {name:<28} ${monthly:>10,.0f} ${per_query:>8.4f} {ptype:>8}{marker}")
    
    # Crossover analysis
    print(f"\n  INSIGHT:")
    api_cheapest = min(r for r in results if r[3] == "API")
    self_cheapest = min(r for r in results if r[3] == "Self")
    
    if api_cheapest[0] < self_cheapest[0]:
        print(f"    At {queries_per_day:,} queries/day: API is cheaper (no fixed GPU cost)")
        # Find crossover
        for mult in [2, 5, 10, 20, 50]:
            self_cost = max((queries_per_day * mult * avg_tokens * 30 / 1_000_000) * 0.05, 2400)
            api_cost = (queries_per_day * mult * avg_tokens * 30 / 1_000_000) * 0.15
            if self_cost < api_cost:
                print(f"    Self-hosted becomes cheaper at ~{queries_per_day * mult:,} queries/day")
                break
    else:
        print(f"    At {queries_per_day:,} queries/day: Self-hosted is cheaper")


# ---------- TEST 3 SCALES ----------

print("PLATFORM COST CALCULATOR")
print("=" * 60)

for volume, label in [(100, "Startup MVP"), (10_000, "Growing Product"), (1_000_000, "Enterprise Scale")]:
    print(f"\n  📊 SCENARIO: {label}")
    print(f"  {'=' * 55}")
    platform_cost_calculator(volume)
    print()

print("💡 Low volume: API is cheapest (no fixed GPU costs).")
print("💡 High volume: self-hosted wins (GPU runs 24/7 anyway).")
print("💡 Crossover: typically around 100K-500K queries/day.")
print("💡 GPT-4o-mini is INSANELY cheap. Hard to beat at low volume.")

---# 7️⃣ Production Health Check Wrapper**Analogy:** A car dashboard. You don't drive without seeing speed, fuel, and engine temperature. This wrapper adds a "dashboard" to every LLM call: latency, tokens, cost, and alerts.**What we build:** A drop-in wrapper that monitors every call. Use it everywhere.

In [ ]:
# ============================================================
# PRODUCTION WRAPPER: Monitor every LLM call automatically
# Drop this into any project. Instant observability.
# ============================================================

class ProductionLLM:
    """
    Drop-in replacement for raw OpenAI calls.
    Automatically tracks: latency, tokens, cost, errors.
    Provides: health check, dashboard, alerts.
    """
    
    PRICES = {"gpt-4o": 5.0, "gpt-4o-mini": 0.15, "gpt-4": 30.0, "gpt-3.5-turbo": 0.50}
    
    def __init__(self, daily_budget=50.0, latency_alert_ms=5000):
        self.client = OpenAI()
        self.calls = []
        self.errors = []
        self.budget = daily_budget
        self.latency_alert = latency_alert_ms
    
    def chat(self, prompt, model="gpt-4o-mini", temperature=0, system=None):
        """Make an LLM call with full monitoring."""
        
        messages = []
        if system: messages.append({"role":"system","content":system})
        messages.append({"role":"user","content":prompt})
        
        start = time.time()
        try:
            r = self.client.chat.completions.create(
                model=model, messages=messages, temperature=temperature)
            latency = (time.time() - start) * 1000
            tokens = r.usage.total_tokens
            cost = tokens / 1_000_000 * self.PRICES.get(model, 1.0)
            answer = r.choices[0].message.content.strip()
            
            self.calls.append({
                "model": model, "tokens": tokens,
                "cost": cost, "latency_ms": latency,
                "success": True, "timestamp": time.time(),
            })
            
            # Alerts
            if latency > self.latency_alert:
                print(f"  ⚠️ SLOW: {latency:.0f}ms > {self.latency_alert}ms threshold")
            
            total_cost = sum(c["cost"] for c in self.calls)
            if total_cost > self.budget * 0.8:
                print(f"  ⚠️ BUDGET: ${total_cost:.4f} > 80% of ${self.budget}")
            
            return answer
            
        except Exception as e:
            latency = (time.time() - start) * 1000
            self.errors.append({"error": str(e), "model": model, "latency_ms": latency})
            self.calls.append({"model": model, "success": False, "timestamp": time.time()})
            raise
    
    def health_check(self):
        """Quick health check: can we reach the LLM?"""
        try:
            start = time.time()
            self.chat("Say OK", model="gpt-4o-mini")
            ms = (time.time() - start) * 1000
            return {"healthy": True, "latency_ms": ms}
        except Exception as e:
            return {"healthy": False, "error": str(e)}
    
    def dashboard(self):
        """Print monitoring dashboard."""
        if not self.calls:
            print("  No calls recorded."); return
        
        successful = [c for c in self.calls if c.get("success", True)]
        total_tokens = sum(c.get("tokens", 0) for c in successful)
        total_cost = sum(c.get("cost", 0) for c in successful)
        latencies = sorted([c.get("latency_ms", 0) for c in successful if c.get("latency_ms")])
        error_rate = len(self.errors) / max(len(self.calls), 1) * 100
        
        print(f"\n  ┌─────────────────────────────────────────┐")
        print(f"  │        PRODUCTION LLM DASHBOARD          │")
        print(f"  ├─────────────────────────────────────────┤")
        print(f"  │  Calls:     {len(self.calls):>6}                      │")
        print(f"  │  Errors:    {len(self.errors):>6} ({error_rate:.1f}%){'  🔴' if error_rate > 2 else '  🟢':>15}│")
        print(f"  │  Tokens:    {total_tokens:>6,}                      │")
        print(f"  │  Cost:      ${total_cost:>8.4f} / ${self.budget:.2f}         │")
        
        if latencies:
            p50 = latencies[len(latencies)//2]
            p95 = latencies[min(int(len(latencies)*0.95), len(latencies)-1)]
            p99 = latencies[min(int(len(latencies)*0.99), len(latencies)-1)]
            print(f"  │  p50:       {p50:>6.0f} ms                    │")
            print(f"  │  p95:       {p95:>6.0f} ms{'  🔴' if p95>5000 else '  🟢':>20}│")
            print(f"  │  p99:       {p99:>6.0f} ms                    │")
        
        print(f"  └─────────────────────────────────────────┘")


# ---------- TEST ----------

print("PRODUCTION LLM WRAPPER")
print("=" * 60)

llm = ProductionLLM(daily_budget=1.0)

# Health check
health = llm.health_check()
print(f"\n  Health check: {'✅ Healthy' if health['healthy'] else '❌ Down'} ({health.get('latency_ms',0):.0f}ms)")

# Simulate production traffic
print("\n  Simulating 10 production calls...")
for q in ["What is 2+2?", "Refund policy?", "Pro plan price?", "Hello!",
           "Compare databases", "Shipping cost?", "Thanks!", "Warranty?",
           "Return electronics?", "Annual discount?"]:
    answer = llm.chat(q)

llm.dashboard()

print("\n💡 Drop this wrapper into ANY project. Instant monitoring.")
print("💡 Health check endpoint: /health (run actual inference, not just ping).")
print("💡 Alert thresholds: p95 > 5s, errors > 2%, cost > 80% budget.")

---# 🏁 Summary — Infrastructure Checklist**Before production:**- [ ] Multi-provider gateway with fallback chain- [ ] Model routing (cheap for simple, expensive for complex)- [ ] Streaming enabled (stream=True)- [ ] Retry with exponential backoff + jitter- [ ] Circuit breaker for provider outages- [ ] Health check endpoint (actual inference, not just ping)- [ ] Cost & latency monitoring on every call**Deployment stack:**```vLLM (engine) → Docker (package) → K8s (scale) → LiteLLM (gateway) → Client```**Cost optimization priority:**1. Route simple queries to cheap models (biggest savings)2. Enable semantic caching (30-40% hit rate typical)3. Shorten outputs (set max_tokens)4. Batch independent calls5. Self-host at high volume (>100K queries/day)